In [153]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
from torch.utils.data import Dataset, DataLoader
import torchmetrics
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import gc

In [154]:
device = 'cuda'

In [155]:
grammar = [
    [('B', 1)], #state 0
    [("T", 2), ("P", 3)], # state 1
    [("S", 2), ("X", 4)], #state 2
    [("T", 3), ("V", 5)],    #state 3
    [("X", 3), ("S", 6)],    #State 4
    [("P", 4), ("V", 6)],    # state 5
     [("E", None)]   # state 6
]

In [156]:
import random

def generate_string(grammar):
    state = 0
    result = ""
    while True:
        letter, state = random.choice(grammar[state])
        #print(letter,state)
        result +=letter
        if state is None:
            break
    return result


In [157]:
def generate_embedded_string(reber_str):
    rnd_choice = random.choice(["T", "P"])
    return 'B'+ rnd_choice + reber_str +  rnd_choice + 'E'

In [158]:
generate_string(grammar)

'BPTVPSE'

In [159]:
embeded_reber = [generate_string(grammar) for x in range(10000)]

In [160]:
x_embedded_strs = [generate_embedded_string(x) for x in embeded_reber]
y_embedded_strs = [1] * 10000

In [161]:
x_embedded_strs[0], embeded_reber[0]

('BTBTSSSXSETE', 'BTSSSXSE')

In [162]:
embedded_strs_wrong = generate_string(grammar)
embeded_reber_wrong = [generate_embedded_string(generate_string(grammar)) for _ in range(10000)]
embeded_reber_wrong[0], embedded_strs_wrong

('BTBTSSXXVVETE', 'BPTVPSE')

In [163]:
def corrupt_string(s):
    length = len(s)
    list_of_letters = ['B', 'E', 'P', 'S', 'T', 'V', 'X']
    rnd_position = random.choice(range(0,length))
    list_of_letters.remove(s[rnd_position])
    rnd_letter = random.choice(list_of_letters)
    s_nou = s[:rnd_position] + rnd_letter + s[rnd_position+1:]
    return s_nou


In [164]:
x_corrupt_strs = [corrupt_string(s) for s in embeded_reber_wrong]
y_corrupt_string =[0] * 10000

In [165]:
X_embedded_grammars= x_corrupt_strs + x_embedded_strs
y_embedded_grammars = y_corrupt_string + y_embedded_strs

In [166]:
len(X_embedded_grammars)

20000

In [167]:
vocab = sorted(set("".join(X_embedded_grammars)))
print(vocab)

['B', 'E', 'P', 'S', 'T', 'V', 'X']


In [168]:
char_to_id = {char : index + 1 for index, char in enumerate(vocab)}
print(char_to_id)
id_to_char = {index + 1: char for index,char in enumerate(vocab)}
print(id_to_char)

{'B': 1, 'E': 2, 'P': 3, 'S': 4, 'T': 5, 'V': 6, 'X': 7}
{1: 'B', 2: 'E', 3: 'P', 4: 'S', 5: 'T', 6: 'V', 7: 'X'}


In [169]:
def encode_text(text):
    return torch.tensor([char_to_id[char] for char in text])

def decode_text(char_ids):
    return "".join([id_to_char[char_id.item()] for char_id in char_ids])


In [170]:
class CharDataset(Dataset):
    def __init__(self, text, target) -> None:
        self.encoded_text = [encode_text(txt) for txt in text]
        self.target=target

    def __getitem__(self, idx ):
        return self.encoded_text[idx], self.target[idx]

    def __len__(self):
        return (len(self.encoded_text))

In [171]:
from torch.nn.utils.rnn import pad_sequence

def collate_fn(batch):
    seqs, labels = zip(*batch)
    lengths = torch.tensor([len(s) for s in seqs])
    X = pad_sequence(seqs, batch_first=True, padding_value=0)
    y = torch.tensor(labels, dtype=torch.float32)
    return X, lengths, y

In [172]:
pairs = list(zip(X_embedded_grammars, y_embedded_grammars))
random.shuffle(pairs)
X_shuf, y_shuf = map(list, zip(*pairs))

train_set = CharDataset(X_shuf[:16000], y_shuf[:16000])
validation_set = CharDataset(X_shuf[16000:18000], y_shuf[16000:18000])
test_set = CharDataset(X_shuf[18000:], y_shuf[18000:])

In [173]:
batch_size = 64

train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)
validation_loader =DataLoader(validation_set, batch_size=batch_size, collate_fn=collate_fn)
test_loader = DataLoader(test_set, batch_size=batch_size, collate_fn=collate_fn)

In [174]:
def evaluate_tm(model, data_loader, metric):
    model.eval()
    metric.reset()
    with torch.no_grad():
        for X_batch, lengths, y_batch in data_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch, lengths)
            metric.update(torch.sigmoid(y_pred), y_batch.long())
    return metric.compute()

def train(model, optimizer, loss_fn, metric, train_loader, valid_loader,
          n_epochs, patience=2, factor=0.5, epoch_callback=None):
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="max", patience=patience, factor=factor)
    history = {"train_losses": [], "train_metrics": [], "valid_metrics": []}
    for epoch in range(n_epochs):
        total_loss = 0.0
        metric.reset()
        model.train()
        if epoch_callback is not None:
            epoch_callback(model, epoch)
        for index, (X_batch, lengths, y_batch) in enumerate(train_loader):
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch, lengths)
            loss = loss_fn(y_pred, y_batch)
            total_loss += loss.item()
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
            metric.update(torch.sigmoid(y_pred), y_batch.long())
            train_metric = metric.compute().item()
            print(f"\rBatch {index + 1}/{len(train_loader)}", end="")
            print(f", loss={total_loss/(index+1):.4f}", end="")
            print(f", {train_metric=:.2%}", end="")
        history["train_losses"].append(total_loss / len(train_loader))
        history["train_metrics"].append(train_metric)
        val_metric = evaluate_tm(model, valid_loader, metric).item()
        history["valid_metrics"].append(val_metric)
        scheduler.step(val_metric)
        print(f"\rEpoch {epoch + 1}/{n_epochs},                      "
              f"train loss: {history['train_losses'][-1]:.4f}, "
              f"train metric: {history['train_metrics'][-1]:.2%}, "
              f"valid metric: {history['valid_metrics'][-1]:.2%}")
    return history

In [175]:
class EmbeddedReberClassifier(nn.Module):
    def __init__(self, vocab_size, n_layers=2, embed_dim=20, hidden_dim=128, dropout=0.1):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.gru = nn.GRU(embed_dim, hidden_dim, num_layers=n_layers, batch_first=True, dropout=dropout)
        self.output = nn.Linear(hidden_dim, 1)

    def forward(self, X, lengths):
        emb = self.embed(X)
        packed = pack_padded_sequence(emb, lengths.cpu(), batch_first=True, enforce_sorted=False)
        outputs, h_n = self.gru(packed)
        return self.output(h_n[-1]).squeeze(-1)

In [176]:
model = EmbeddedReberClassifier(vocab_size=len(vocab) + 1).to(device)
optimizer = torch.optim.NAdam(model.parameters(), lr=1e-3)
loss_fn = nn.BCEWithLogitsLoss()
metric = torchmetrics.Accuracy(task="binary").to(device)

history = train(model, optimizer, loss_fn, metric,
                train_loader, validation_loader, n_epochs=20)

test_acc = evaluate_tm(model, test_loader, metric).item()
print(f"Test accuracy: {test_acc:.2%}")

Epoch 1/20,                      train loss: 0.6084, train metric: 64.50%, valid metric: 81.45%
Epoch 2/20,                      train loss: 0.3696, train metric: 83.31%, valid metric: 97.30%
Epoch 3/20,                      train loss: 0.1350, train metric: 96.34%, valid metric: 98.45%
Epoch 4/20,                      train loss: 0.0683, train metric: 98.54%, valid metric: 99.25%
Epoch 5/20,                      train loss: 0.0308, train metric: 99.32%, valid metric: 100.00%
Epoch 6/20,                      train loss: 0.0008, train metric: 100.00%, valid metric: 100.00%
Epoch 7/20,                      train loss: 0.0003, train metric: 100.00%, valid metric: 100.00%
Epoch 8/20,                      train loss: 0.0002, train metric: 100.00%, valid metric: 100.00%
Epoch 9/20,                      train loss: 0.0001, train metric: 100.00%, valid metric: 100.00%
Epoch 10/20,                      train loss: 0.0001, train metric: 100.00%, valid metric: 100.00%
Epoch 11/20,                

In [180]:
def predict(model, strings):
    model.eval()
    X, lengths, _ = collate_fn([(encode_text(s), 0.0) for s in strings])
    with torch.no_grad():
        return torch.sigmoid(model(X.to(device), lengths)).cpu()

tests = [
    ("BTBPVVETE",                1),  # valid, scurt
    ("BPBTSXXVPSEPE",            1),  # valid, exemplul din carte
    ("BTBTSSSSSXXTTTVPXVVETE",   1),  # valid, lung
    ("BTBPVVEPE",                0),  # wrapper nepotrivit (T...P)
    ("BPBTSSSSSXXTTTVPXVVETE",   0),  # wrapper nepotrivit, LUNG: testul greu
    ("BTBTXXVVSETE",             0),  # greșeală în inner (S după starea 6)
    ("BPBPTVPXVPEPE",            0),  # greșeală în inner (E din starea 4)
]

strings = [s for s, _ in tests]
probs = predict(model, strings)
for (s, label), p in zip(tests, probs):
    pred = int(p > 0.5)
    mark = "OK" if pred == label else "GRESIT"
    print(f"{s:<25} P(valid)={p:.3f}  asteptat={label}  {mark}")

BTBPVVETE                 P(valid)=1.000  asteptat=1  OK
BPBTSXXVPSEPE             P(valid)=1.000  asteptat=1  OK
BTBTSSSSSXXTTTVPXVVETE    P(valid)=1.000  asteptat=1  OK
BTBPVVEPE                 P(valid)=0.001  asteptat=0  OK
BPBTSSSSSXXTTTVPXVVETE    P(valid)=0.000  asteptat=0  OK
BTBTXXVVSETE              P(valid)=0.000  asteptat=0  OK
BPBPTVPXVPEPE             P(valid)=0.000  asteptat=0  OK
